In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock
import pandas as pd
import re
import time
import pandas as pd
import nltk
import spacy
import numpy as np
from collections import Counter

# Load English tokenizer, POS tagger, parser, NER, and word vectors
nlp = spacy.load("en_core_web_sm")

# Define a word list using nltk corpus
english_vocab = set(w.lower() for w in nltk.corpus.words.words())

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_22"),
    os.getenv("OPENAI_KEY_23"),
    os.getenv("OPENAI_KEY_24"),
    os.getenv("OPENAI_KEY_25")
]

In [ ]:
pd.set_option('display.max_colwidth', 100)

In [ ]:
executor = ThreadPoolExecutor(max_workers=25)

In [ ]:
transcript = "../runs/2008-09-26-debate-preclean.txt"

In [ ]:
def process_transcript(file_path):
    # Open the file.
    with open(file_path, "r") as file:
        # Read the content.
        content = file.read()

    # Split the content into lines.
    lines = content.split("\n")

    # Initialize a variable to keep track of the current speaker.
    current_speaker = None

    # Initialize a variable to keep track of the speaker's turn.
    turn = 0

    # Prepare a list to store the rows of the DataFrame.
    rows = []

    # Initialize a variable to keep track of the current line.
    current_line = ""

    # Go through each line.
    for line in lines:
        # Check if line contains the speaker's name and what they say.
        if ": " in line:
            # Split the line into the speaker's name and what they say.
            speaker, line = line.split(": ", 1)

            # If the speaker has changed, add the previous turn to the list of rows and start a new turn.
            if speaker != current_speaker:
                if current_speaker is not None:
                    rows.append([turn, current_speaker, current_line.strip()])
                turn += 1
                current_speaker = speaker
                current_line = line
            else:
                # If the speaker hasn't changed, add the line to the current turn.
                current_line += " " + line
        else:
            # If the line doesn't contain a speaker, add it to the current turn.
            current_line += " " + line

    # Add the last turn to the list of rows.
    rows.append([turn, current_speaker, current_line.strip()])

    # Create a DataFrame from the list of rows.
    df = pd.DataFrame(rows, columns=["Turn", "Speaker", "Content"])

    # Return the DataFrame.
    return df


In [ ]:
transcript_dataframe = process_transcript(transcript)

In [ ]:
def make_scoreboard(df):
    # Initialize the data frame with default values
    df['Total_Score'] = 0.0
    df['Base_Score'] = 0
    df['EPL_Multiplier'] = 0.0
    df['Sentence_Count'] = 0
    df['Word_Count'] = 0
    df['Word_Count_Bonus'] = 0
    df['Named_Entity_Count'] = 0
    df['E'] = 0.0
    df['P'] = 0.0
    df['L'] = 0.0
    df['E_Expl'] = ""
    df['P_Expl'] = ""
    df['L_Expl'] = ""
    df['Cumulative_Speaker_Turn'] = 0
    df['Cumulative_Speaker_Base_Score'] = 0
    df['Cumulative_Speaker_Total_Score'] = 0.0

    # Get list of unique speakers
    speakers = df['Speaker'].unique()

    # Initialize counters for each speaker
    speaker_turns = Counter()
    speaker_base_scores = Counter()

    for idx, row in df.iterrows():
        content = row['Content']
        speaker = row['Speaker']

        # Sentence Count - sentences ending with '.', '?', '!'
        sentence_count = len(nltk.sent_tokenize(content))

        # Word Count - total number of words
        words = nltk.word_tokenize(content)
        word_count = len(words)

        # Word Count Bonus - unique English words
        unique_english_words = len(set(w for w in words if w.lower() in english_vocab))

        # Named Entity Count
        doc = nlp(content)
        named_entity_count = len(doc.ents)

        # Update Base Score
        base_score = sentence_count + word_count + unique_english_words + named_entity_count

        # Update cumulative speaker scores
        speaker_turns[speaker] += 1
        speaker_base_scores[speaker] += base_score

        # Update DataFrame
        df.loc[idx, 'Sentence_Count'] = sentence_count
        df.loc[idx, 'Word_Count'] = word_count
        df.loc[idx, 'Word_Count_Bonus'] = unique_english_words
        df.loc[idx, 'Named_Entity_Count'] = named_entity_count
        df.loc[idx, 'Base_Score'] = base_score
        df.loc[idx, 'Cumulative_Speaker_Turn'] = speaker_turns[speaker]
        df.loc[idx, 'Cumulative_Speaker_Base_Score'] = speaker_base_scores[speaker]
    
    return df


In [ ]:
scoreboard = make_scoreboard(transcript_dataframe)

In [ ]:
scoreboard.shape

In [ ]:
def parse_content(content):
    try:
        # Try to parse the string as a JSON
        parsed_content = json.loads(content.replace("'", "\""))  # replace single quotes with double quotes
    except json.JSONDecodeError:
        # If it fails, parse it using regular expressions
        matches = re.findall(r'(\w+_?\w*): ([\d\.]+|".*?")', content)
        parsed_content = {key: value.strip('""') for key, value in matches}
    
    return parsed_content


In [ ]:
def prompt_epl(row, prev_row=None):
    if not row['Content']: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Evaluate the following snippet for Ethos (E), Pathos (P), and Logos (L). (Scale: 0.0 - 9.9) For non-zero score only: provide a brief explanation, otherwise leave empty string. It should be equally difficult to score 0.0 as it would be to score 9.9. [Snippet: '{row['Content']}'] Return a dict formatted like this: 'E: x.x, P: y.y, L: z.z, E_Expl: "", P_Expl: "", L_Expl: "". Explanations must be 1 sentence long."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt


In [ ]:
def get_epl(input_dataframe, output_file_full, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startturn is not None and endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with 'startturn' and 'endturn'. Use 'runcount' alone or with either 'startturn' and 'endturn'.")

    # Determine start and end points for analysis
    start = startturn if startturn is not None else 0
    end = endturn if endturn is not None else len(input_dataframe)

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(input_dataframe))

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(input_dataframe):
        end = len(input_dataframe)
        print(f"Warning: The end point exceeded the length of the dataframe. It has been adjusted to the last turn: {end}")

    # Create locks for writing to file and dataframe
    lock1 = Lock()
    lock2 = Lock()

    # Use a ThreadPoolExecutor to process the lines concurrently
    with ThreadPoolExecutor(max_workers=25) as executor:
        # Assign API keys cyclically
        futures = [executor.submit(threadlock, i, output_file_full, input_dataframe, lock1, lock2, open_ai_keys[i % len(open_ai_keys)]) for i in range(start, end)]
        
        # Ensure all threads have finished
        for future in as_completed(futures):
            pass

    # Sort DataFrame by 'Turn' after all threads are completed
    input_dataframe.sort_values('Turn', inplace=True)
    input_dataframe.reset_index(drop=True, inplace=True)

    # Recalculate the cumulative scores
    for idx in range(1, len(input_dataframe)):
        if input_dataframe.loc[idx, 'Speaker'] == input_dataframe.loc[idx - 1, 'Speaker']:
            input_dataframe.loc[idx, 'Cumulative_Speaker_Turn'] = input_dataframe.loc[idx - 1, 'Cumulative_Speaker_Turn'] + 1
            input_dataframe.loc[idx, 'Cumulative_Speaker_Base_Score'] = input_dataframe.loc[idx - 1, 'Cumulative_Speaker_Base_Score'] + input_dataframe.loc[idx, 'Base_Score']
            input_dataframe.loc[idx, 'Cumulative_Speaker_Total_Score'] = input_dataframe.loc[idx - 1, 'Cumulative_Speaker_Total_Score'] + input_dataframe.loc[idx, 'Total_Score']
        else:
            input_dataframe.loc[idx, 'Cumulative_Speaker_Turn'] = 1
            input_dataframe.loc[idx, 'Cumulative_Speaker_Base_Score'] = input_dataframe.loc[idx, 'Base_Score']
            input_dataframe.loc[idx, 'Cumulative_Speaker_Total_Score'] = input_dataframe.loc[idx, 'Total_Score']

    return input_dataframe


In [ ]:
scored_test_df = get_epl(transcript_dataframe, "2008-test-9.json", startturn=1, endturn=5)

In [ ]:
scored_test_df.iloc[0:4]